# Notebook for computing **TRUE DTW Similarities for Rome and Porto** 

In [ ]:
# Importing nescessary modules
import os, sys
import shutil

def find_project_root(target_folder="masteroppgave"):
    """Find the absolute path of a folder by searching upward."""
    currentdir = os.path.abspath("__file__")  # Get absolute script path
    while True:
        if os.path.basename(currentdir) == target_folder:
            return currentdir  # Found the target folder
        parentdir = os.path.dirname(currentdir)
        if parentdir == currentdir:  # Stop at filesystem root
            return None
        currentdir = parentdir  # Move one level up

project_root = find_project_root("masteroppgave")

if project_root:
    sys.path.append(project_root)
    print(f"Project root found: {project_root}")
else:
    raise RuntimeError("Could not find 'masteroppgave' directory")

from utils.helpers import file_handler as fh
from utils.helpers import metafile_handler as mfh
from utils.similarity_measures import dtw
import numpy as np

## FUNCTIONS

In [ ]:
def deleteFile(file_name: str, folder_name: str) -> None:
    file_path = os.path.join(folder_name, file_name)
    try:
        if os.path.isfile(file_path) or os.path.islink(file_path):
            os.unlink(file_path)
        elif os.path.isdir(file_path):
            shutil.rmtree(file_path)
    except Exception as e:
        print("Failed to remove %s. Reason: %s" % (file_path, e))
        
        

def generate_dtw_similarities(
    data_folder: str, meta_file: str, file_name: str, similarities_output_folder: str
):
    deleteFile(file_name, similarities_output_folder) # Delete file if it exists before running

    files = mfh.read_meta_file(meta_file)

    trajectories = fh.load_trajectory_files(files, data_folder)

    df = dtw.cy_dtw(trajectories)

    df.to_csv(os.path.join(similarities_output_folder, file_name))


def generate_parallell_dtw_similarities(
    data_folder: str, meta_file: str, file_name: str, similarities_output_folder: str
):
    deleteFile(file_name, similarities_output_folder)

    files = mfh.read_meta_file(meta_file)
    trajectories = fh.load_trajectory_files(files, data_folder)

    df = dtw.cy_dtw_pool(trajectories)
    df.to_csv(os.path.join(similarities_output_folder, file_name))

# DTW SIMILARITIES FOR ROME


In [ ]:
ROME_DATA_FOLDER = "../../../dataset/rome/output/"
ROME_SIMILARITY_VALUES_RESULT_FOLDER = "../../../results_true/similarity_values/rome/dtw"

SIZES = np.arange(750, 3050, 50)  # Generates [0.5, 1.0, 1.5, ..., 5.5]


for size in SIZES:
    ROME_DATA_META_FILE = f"{ROME_DATA_FOLDER}META-{size}.txt"
    ROME_DTW_FILENAME =  f"rome-dtw-{size}.csv"
    
    print("Generating DTW similarities for Rome with size: ", size)
    
    generate_parallell_dtw_similarities(
        ROME_DATA_FOLDER, ROME_DATA_META_FILE, ROME_DTW_FILENAME, ROME_SIMILARITY_VALUES_RESULT_FOLDER
    )

# DTW SIMILARITIES FOR PORTO


In [ ]:
PORTO_DATA_FOLDER  = "../../../dataset/porto/output/"
PORTO_SIMILARITY_VALUES_RESULT_FOLDER = "../../../results_true/similarity_values/porto/dtw"

SIZES = np.arange(100, 1500, 100)  # Generates [0.5, 1.0, 1.5, ..., 5.5]

for size in SIZES:
    PORTO_DATA_META_FILE = f"{PORTO_DATA_FOLDER}/META-{size}.txt"
    PORTO_DTW_FILENAME =  f"porto-dtw-{size}.csv"
    
    print("Generating DTW similarities for Porto with size: ", size)
    
    generate_parallell_dtw_similarities(
        PORTO_DATA_FOLDER,
        PORTO_DATA_META_FILE,
        PORTO_DTW_FILENAME,
        PORTO_SIMILARITY_VALUES_RESULT_FOLDER,
    )  